# 序列：GRU 與長度處理

對應原作業：延伸 Day 63 的序列討論。這是現代 PyTorch 練習；原始作業保留在 `../../homework/`。

從上到下執行即可。預設使用 CPU、合成資料，不需下載外部資料；改為真實資料時，先決定資料粒度與切分方式。

## 序列資料
同一實體的時間資料不能隨機跨訓練／驗證；先建立預測時點，再處理變長序列。GRU 適合做輕量序列基準；更長或需要預訓練時再評估 Transformer。

In [ ]:
import torch
from torch import nn
from torch.nn.utils.rnn import pack_padded_sequence

torch.manual_seed(42)
sequences = torch.randn(3, 5, 4)  # [batch, max_length, features]
lengths = torch.tensor([5, 3, 4])
for i, length in enumerate(lengths):
    sequences[i, length:] = 0

class SequenceClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.gru = nn.GRU(input_size=4, hidden_size=8, batch_first=True)
        self.head = nn.Linear(8, 2)

    def forward(self, x, length):
        packed = pack_padded_sequence(x, length.cpu(), batch_first=True, enforce_sorted=False)
        _, hidden = self.gru(packed)
        return self.head(hidden[-1])

model = SequenceClassifier()
logits = model(sequences, lengths)
target = torch.tensor([0, 1, 0])
loss = nn.CrossEntropyLoss()(logits, target)
loss.backward()
print(logits.shape, loss.item())
assert logits.shape == (3, 2)
